# v10 B300 measured core — the regime knee-hunt (sm_103, root/bare-metal)

The paper's headline + the one paid step. Carries the v9-Task-1 method (clock-lock + L2-flush +
counter-free `L2!` + ncu) to **B300/sm_103** and asks the T3 question: **does a bandwidth-bound decode
regime exist past B300's ~126 MB L2?** Sweep N_k far past the L2 crossing for all three KV formats
(FP16 / FP8 / **NVFP4**) and watch %HBM + ncu L2-hit-rate. **Knee found** → NVFP4's byte cut converts
(a bandwidth result). **Flat per-CTA cap to 1M tokens** → the stronger, more surprising result
(per-CTA-bound on sm_103). Either sign is publishable.

**TIERED for cost control:** run §4 smoke + §5 arch-measure + Tier-1 (→256K) first (~short); escalate
to Tier-2 (→524K) + Tier-3 (→1M+) only if the cheap tier looks good. **Build first on a B200 dev-rung**
(`docs/v10-b300-runbook.md`), then the B300 record. Gencode auto-detects sm_103 (CUDA 12.9 box).

## 0. Dependencies + GPU (venv-safe)

In [1]:
import os, sys, subprocess

def pip(*pkgs, extra=()):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs, *extra], check=True)

try:
    has_gpu = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
except FileNotFoundError:
    has_gpu = False
if not has_gpu:
    raise SystemExit('No GPU. FIX: rent a B200 (sm_100) or B300 (sm_103) on vast.ai — privileged/bare-metal for ncu.')

# matplotlib for the decisive plot (the one new dep vs other gates); numpy before torch.
pip('ninja', 'pytest', 'numpy', 'matplotlib')

try:
    import torch
    cuda_ok = torch.cuda.is_available()
except ImportError:
    torch, cuda_ok = None, False
if not cuda_ok:
    # Blackwell (B200 sm_100 / B300 sm_103) has no SASS in the cu124 wheel; nightly cu129 covers both.
    # (Skipped entirely if your image already ships a Blackwell-capable torch.)
    pip('--pre', 'torch', extra=('--index-url', 'https://download.pytorch.org/whl/nightly/cu129'))
    raise SystemExit('Installed Blackwell torch (nightly cu129). RESTART the kernel + re-run from the top.')

os.environ['PATH'] = os.path.dirname(sys.executable) + os.pathsep + os.environ.get('PATH', '')
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| cap', torch.cuda.get_device_capability())
!nvidia-smi --query-gpu=name,compute_cap,clocks.current.sm,clocks.max.sm --format=csv

torch 2.10.0+cu128 | cuda 12.8 | cap (10, 0)
name, compute_cap, clocks.current.sm [MHz], clocks.max.sm [MHz]
NVIDIA B200, 10.0, 600 MHz, 1965 MHz


## 1. Get the repo

In [2]:
REPO_URL = 'https://github.com/gkienpham-cmd/flashattention-cuda.git'  # public; plain clone works
import os, sys, subprocess
if os.path.basename(os.getcwd()) != 'flashattention-cuda':
    if not os.path.isdir('flashattention-cuda'):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('flashattention-cuda')
subprocess.run(['git', 'pull', 'origin', 'main'])
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('cwd', os.getcwd())

Cloning into 'flashattention-cuda'...


Already up to date.
cwd /flashattention-cuda


From https://github.com/gkienpham-cmd/flashattention-cuda
 * branch            main       -> FETCH_HEAD


## 2. Lock clocks (root / bare-metal — works on a privileged B200/B300, unlike T4 containers)

In [3]:
# Lock clocks so wall-times are comparable across runs. NEEDS ROOT (vast.ai bare-metal). On free
# Colab this prints a LOUD warning and continues — the counter-free %HBM/eff_bw sweep still runs, but
# cross-run wall-times are confounded (exactly the C12 problem). Reset is in the last cell.
from bench.regime import lock_clocks
ok, sm_mhz, mem_mhz, throttle = lock_clocks()
print('locked =', ok, '| sm =', sm_mhz, 'MHz | mem =', mem_mhz, 'MHz | throttle:', throttle or 'none')
if not ok:
    print('\\n>>> Running UNLOCKED: trust %HBM / eff_bw / L2! (intra-run, clock-robust);')
    print('>>> do NOT compare absolute us/tok across runs. For the publishable verdict, use a root T4.')

# !!! CLOCKS NOT LOCKED (need root / bare-metal) — cross-run wall-times are CONFOUNDED.
#     (CalledProcessError: run on a root T4 to lock; the counter-free sweep still runs.)
locked = False | sm = 120 MHz | mem = 0 MHz | throttle: none
\n>>> Running UNLOCKED: trust %HBM / eff_bw / L2! (intra-run, clock-robust);
>>> do NOT compare absolute us/tok across runs. For the publishable verdict, use a root T4.


## 3. PRE-FLIGHT SMOKE TEST (fail-fast — validates the sm_103 build before any paid sweep)

In [4]:
# Build all three backends (this is where the auto-detected compute_103 gencode is exercised — if the
# box has CUDA 12.9 it compiles; if it errors here, STOP and fix before burning sweep time), then a
# tiny correctness check + a 2-point sweep to confirm they RUN. < 1 min. Green here = box is good.
import glob, os, shutil, subprocess, sys
from bindings.load import build_kernel
for name in ('v8_gqa_ss', 'v9_fp8', 'v10_nvfp4'):
    for dd in glob.glob(os.path.expanduser(f'~/.cache/torch_extensions/*/fa_{name}')):
        if not glob.glob(os.path.join(dd, '*.so')):
            shutil.rmtree(dd, ignore_errors=True); print('cleaned stale build:', dd)
mods = {n: build_kernel(n) for n in ('v8_gqa_ss', 'v9_fp8', 'v10_nvfp4')}
print('built:', {n: m is not None for n, m in mods.items()})
# correctness smoke: the cheap square-reduction cases for v10 (fast; full gate is the v10 gate nb)
r = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_correctness.py',
                    '-k', 'v10_nvfp4_square or v9_fp8_square', '-q'], capture_output=True, text=True)
print(r.stdout[-1200:])
assert 'passed' in r.stdout and 'failed' not in r.stdout, 'SMOKE CORRECTNESS FAILED — stop and debug'
# run smoke: one tiny sweep point per backend (confirms each kernel launches on this arch)
from bench.regime import sweep
for be in ('v8_gqa_ss', 'v9_fp8', 'v10_nvfp4'):
    _ = sweep(be, [2048], batches=[1], head_dims=[128], h_kvs=[1], warmup=3, iters=10, l2_flush=False)
print('\nSMOKE GREEN — build + correctness + launch all OK. Safe to run the paid sweeps.')

[1/3] c++ -MMD -MF binding.o.d -DTORCH_EXTENSION_NAME=fa_v8_gqa_ss -DTORCH_API_INCLUDE_EXTENSION_H -I/venv/main/lib/python3.12/site-packages/nvidia/cublas/include -I/venv/main/lib/python3.12/site-packages/nvidia/cuda_cupti/include -I/venv/main/lib/python3.12/site-packages/nvidia/cuda_nvrtc/include -I/venv/main/lib/python3.12/site-packages/nvidia/cuda_runtime/include -I/venv/main/lib/python3.12/site-packages/nvidia/cudnn/include -I/venv/main/lib/python3.12/site-packages/nvidia/cufft/include -I/venv/main/lib/python3.12/site-packages/nvidia/cufile/include -I/venv/main/lib/python3.12/site-packages/nvidia/curand/include -I/venv/main/lib/python3.12/site-packages/nvidia/cusolver/include -I/venv/main/lib/python3.12/site-packages/nvidia/cusparse/include -I/venv/main/lib/python3.12/site-packages/nvidia/cusparselt/include -I/venv/main/lib/python3.12/site-packages/nvidia/nccl/include -I/venv/main/lib/python3.12/site-packages/nvidia/nvjitlink/include -I/venv/main/lib/python3.12/site-packages/nvidia

## 4. ARCH MEASUREMENT — fill the speculative B300 constants (esp. the unknown L2 size)

In [ ]:
# roofline/archs.py B300 has l2_mb=None + speculative clock/smem/SM. Measure them from the device so
# the L2-crossing math (and the paper's arch table) is REAL, not assumed. Prints the exact archs.py
# lines to paste back. THE key number: L2 size (assumed ~126 MB; confirm or overturn here).
import torch
p = torch.cuda.get_device_properties(0)
l2_bytes = getattr(p, 'L2_cache_size', None)
smem_sm  = getattr(p, 'shared_memory_per_multiprocessor', None) or getattr(p, 'max_shared_memory_per_multiprocessor', None)
print(f'device      : {p.name}  (sm_{p.major}{p.minor})')
print(f'num_sm      : {p.multi_processor_count}')
print(f'total HBM   : {p.total_memory/1e9:.1f} GB')
print(f'L2 size     : {l2_bytes/1e6:.1f} MB  ({l2_bytes} bytes)' if l2_bytes else 'L2 size     : (not exposed by torch — read from `nvidia-smi -q` / cudaDeviceGetAttribute)')
print(f'smem/SM     : {smem_sm/1024:.0f} KB' if smem_sm else 'smem/SM     : (n/a)')
import subprocess
smi = subprocess.run(['nvidia-smi','--query-gpu=clocks.max.sm,clocks.max.mem','--format=csv,noheader'], capture_output=True, text=True)
print('max clocks  :', smi.stdout.strip())
L2_MB = (l2_bytes/1e6) if l2_bytes else 126.0   # fall back to the ~126 MB assumption for the sweep math
print(f'\n>>> Using L2_MB = {L2_MB:.0f} for the L2-crossing sweep math below.')
print('>>> PASTE INTO roofline/archs.py B300: '
      f'num_sm={p.multi_processor_count}, l2_mb={L2_MB:.0f}'
      + (f', smem_per_sm_kb={smem_sm/1024:.0f}' if smem_sm else '') + ' (+ the measured max SM clock).')

## 5. Roofline framing (record BEFORE the sweep) + the measured L2-crossing points

In [ ]:
# Decode AI = 2G/b, HBM-bound per the model at every N_k — but the model is BLIND to L2. The crossing
# is where the KV working set (2*N_k*d*b, B=1 H_kv=1) exceeds the MEASURED L2. NVFP4 (b=0.5625) holds
# the cache ~3.55x longer than FP16, so its crossing sits at the largest N_k -> the sweep must reach 1M.
from roofline.archs import get_arch
arch = get_arch('sm_103')
print('arch:', arch.name, '| HBM', arch.hbm_bw_gbps/1e3, 'TB/s | NVFP4', arch.fp4_tc_flops/1e15, 'PF | exp', arch.exp_per_s/1e12, 'TExp/s')
print(f'\nL2-capacity crossing (KV = 2*N_k*d*b = {L2_MB:.0f} MB), B=1 H_kv=1:')
for d in (64, 128):
    for b, lab in ((2, 'fp16'), (1, 'fp8'), (0.5625, 'nvfp4')):
        n_cross = int(L2_MB*1e6 / (2*d*b))
        print(f'  d={d:3d} {lab:6s}: N_k ~= {n_cross:8d}  (KV exceeds L2 past this)')
print('\nPREDICTION (per-CTA survivor): %HBM stays ~flat past L2 with L2served=False -> per-CTA-bound on')
print('sm_103, NVFP4 byte cut does NOT convert (capacity+accuracy only). COUNTER (the prize): %HBM CLIMBS')
print('toward the achievable ceiling once N_k crosses L2 -> bandwidth-bound on sm_103 -> NVFP4 wins there.')

## 6. Tier 1 (SHORT, ~1-2 hr) — knee-hunt to 256K, all three KV formats

In [ ]:
# The decisive first look. H_kv=1 (cleanest L2 crossing), d in {64,128}, N_k crossing the ~126 MB L2
# for FP16/FP8 (NVFP4 crossing needs Tier 3). ROWS accumulates across tiers for the plot.
from bench.regime import sweep
ROWS = {}
KV_T1 = [8192, 32768, 131072, 262144]
for be in ('v8_gqa_ss', 'v9_fp8', 'v10_nvfp4'):
    print(f'================ {be} (Tier 1) ================')
    ROWS[be] = sweep(be, KV_T1, batches=[1], head_dims=[64, 128], h_kvs=[1], max_ws_gb=32.0)
print('\nTier 1 done. Look at %HBM vs N_k + any L2! flags before escalating. Plot in §9.')

## 7. Tier 2 (MEDIUM, ~3-4 hr) — extend to 524K (crosses FP8's L2), + large-batch

In [ ]:
# Run only if Tier 1 warrants it. Extends N_k to 524K and adds the large-batch confirmation past L2.
KV_T2 = [524288]
for be in ('v8_gqa_ss', 'v9_fp8', 'v10_nvfp4'):
    print(f'================ {be} (Tier 2: N_k=524288) ================')
    ROWS[be] += sweep(be, KV_T2, batches=[1], head_dims=[64, 128], h_kvs=[1], max_ws_gb=48.0)
print('\n======== large-batch past L2 (N_k=131072, sweep B): does %HBM climb as BH fills 160 SMs? ========')
for be in ('v8_gqa_ss', 'v10_nvfp4'):
    print(f'---- {be} ----')
    _ = sweep(be, [131072], batches=[1, 8, 32, 64, 128], head_dims=[128], h_kvs=[1], max_ws_gb=64.0)

## 8. Tier 3 (OPEN, no cap) — to 1M+ (crosses NVFP4's L2), repeats for noise

In [ ]:
# The full knee-hunt: N_k past NVFP4's L2 crossing (~870K at d=128). This is where NVFP4 either finally
# goes bandwidth-bound or proves per-CTA-bound to a million tokens. Large WS (d128 fp16 @1M = 512 MB);
# 288 GB HBM holds it. Raise max_ws_gb as needed.
KV_T3 = [1048576, 2097152]
for be in ('v8_gqa_ss', 'v9_fp8', 'v10_nvfp4'):
    print(f'================ {be} (Tier 3) ================')
    ROWS[be] += sweep(be, KV_T3, batches=[1], head_dims=[128], h_kvs=[1], max_ws_gb=96.0)
print('\nTier 3 done — the full N_k range captured. Plot below is the paper figure.')

## 9. THE DECISIVE PLOT — %HBM vs N_k, all three KV formats, L2 crossings marked

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, os
fig, ax = plt.subplots(figsize=(9, 5.5))
styles = {'v8_gqa_ss': ('-', 'FP16'), 'v9_fp8': ('--', 'FP8'), 'v10_nvfp4': (':', 'NVFP4')}
for be, (ls, lab) in styles.items():
    for d, color in ((64, 'tab:blue'), (128, 'tab:red')):
        pts = sorted([r for r in ROWS.get(be, []) if r['H_kv'] == 1 and r['d'] == d], key=lambda r: r['N_k'])
        if not pts: continue
        ax.plot([r['N_k'] for r in pts], [r['hbm_pct'] for r in pts], ls, marker='o', color=color,
                label=f'{lab} d={d}')
        for r in pts:
            if r.get('l2_served'): ax.annotate('L2!', (r['N_k'], r['hbm_pct']), fontsize=7, color='green')
ax.axhline(70, color='gray', ls=':', lw=1, label='achievable ceiling (~70%)')
for b, lab, col in ((2,'fp16','black'), (0.5625,'nvfp4','purple')):
    ax.axvline(int(L2_MB*1e6/(2*128*b)), color=col, ls=':', lw=1, label=f'L2 cross d128 {lab}')
ax.set_xscale('log', base=2); ax.set_xlabel('N_k (KV length)'); ax.set_ylabel('% of peak HBM BW')
ax.set_title(f'v10 B300 (sm_103) — decode %HBM vs N_k (B=1, H_kv=1, L2-flushed, L2={L2_MB:.0f}MB)')
ax.legend(fontsize=8, ncol=2); ax.grid(True, which='both', alpha=0.3)
os.makedirs('docs/diagrams', exist_ok=True)
fig.savefig('docs/diagrams/v10-b300-regime.svg', bbox_inches='tight')
fig.savefig('docs/diagrams/v10-b300-regime.png', dpi=110, bbox_inches='tight')
print('saved docs/diagrams/v10-b300-regime.svg'); plt.show()

## 10. ncu cross-check (works on root/privileged B200/B300 — the confound-free settle)

In [7]:
# L2 hit-rate + DRAM% for an L2-resident shape and a past-L2 shape. On a privileged Blackwell box the
# ERR_NVGPUCTRPERM that blocked T4 containers is gone -> these counters confirm (or overturn) the
# counter-free %HBM verdict. Run for v10_nvfp4 past L2 (the headline kernel).
import subprocess, sys
METRICS = ('lts__t_sector_hit_rate.pct,'
           'dram__throughput.avg.pct_of_peak_sustained_elapsed,'
           'lts__throughput.avg.pct_of_peak_sustained_elapsed')
for tag, shape in (('L2-resident N_k=8192', '1,1,8192,128'), ('past-L2 N_k=1048576', '1,1,1048576,128')):
    print(f'\n===== ncu {tag} (v10_nvfp4) =====')
    cmd = ['ncu', '--metrics', METRICS, '--launch-count', '5',
           '--kernel-name', 'regex:(gqa_ss|fp8|nvfp4)', '--target-processes', 'all',
           sys.executable, '-m', 'bench.regime', '--profile', shape, '--backend', 'v10_nvfp4']
    try:
        out = subprocess.run(cmd, capture_output=True, text=True, timeout=900)
        print(out.stdout[-2500:] if out.stdout else '(no stdout)')
        if 'ERR_NVGPUCTRPERM' in (out.stdout + out.stderr):
            print('>>> ncu blocked — instance is NOT privileged. Re-rent a bare-metal/privileged box.')
    except FileNotFoundError:
        print('>>> ncu not installed; apt-get install or use the CUDA -devel image.'); break
    except Exception as e:
        print('>>> ncu failed:', type(e).__name__, e)


===== ncu L2-resident N_k=8192 (v10_nvfp4) =====
>>> ncu not installed; apt-get install or use the CUDA -devel image.


## 10b. Nsight Systems TRACE (container-friendly — works WITHOUT the ncu permission gate)

In [5]:
# `nsys` does timeline/trace profiling via CUPTI and runs in an UNPRIVILEGED container (no
# ERR_NVGPUCTRPERM, unlike ncu). It gives the kernel-time breakdown (corroborates the per-CTA SCHEDULE:
# one partial kernel ~100%% of GPU time + a tiny LSE-merge), but NOT hardware counters — there is no
# L2-hit-rate / DRAM%% here (those need ncu on a privileged/bare-metal box, §10 + runbook). Profiles one
# past-L2 shape for v10_nvfp4.
import subprocess, sys, shutil
if shutil.which('nsys') is None:
    print('nsys not installed. Install (unprivileged-OK): apt-get update && apt-get install -y nsight-systems')
    print('(or it ships with a CUDA -devel image). Then re-run this cell.')
else:
    cmd = ['nsys', 'profile', '-o', '/tmp/v10_nsys', '--force-overwrite', 'true',
           '--stats', 'true', '--trace', 'cuda',
           sys.executable, '-m', 'bench.regime', '--profile', '1,1,1048576,128', '--backend', 'v10_nvfp4']
    out = subprocess.run(cmd, capture_output=True, text=True, timeout=900)
    print((out.stdout + out.stderr)[-3800:])
    print('\n>>> Read the "CUDA GPU Kernel Summary": the partial kernel should be ~100%% of GPU time and the')
    print('>>> merge a sliver -> the per-CTA schedule, confirmed without needing counters. (No L2-hit-rate')
    print('>>> here; that is the ncu deliverable on a privileged box.)')

ninja: no work to do.
# profiled 100 launches of v10_nvfp4 @ B1 H_kv1 N1048576 d128 (for ncu)
Generating '/tmp/nsys-report-d141.qdstrm'
Generated:
    /tmp/v10_nsys.qdstrm
Importer error status: The importer binary and its dependencies were not found.
Unable to retrieve the importer version: skipping importation of the QDSTRM file.


>>> Read the "CUDA GPU Kernel Summary": the partial kernel should be ~100%% of GPU time and the
>>> merge a sliver -> the per-CTA schedule, confirmed without needing counters. (No L2-hit-rate
>>> here; that is the ncu deliverable on a privileged box.)


## 11. Reset clocks (always run before destroying the instance)

In [6]:
from bench.regime import reset_clocks
reset_clocks()

# clocks reset.


## 12. Verdict (fill after the run)

- **Knee found** (%HBM climbs past the L2 crossing, ncu L2-hit-rate drops): decode IS bandwidth-bound
  on sm_103 past L2 → NVFP4's byte cut converts to latency. The conditional T3 bandwidth claim is earned.
- **Flat per-CTA cap to 1M** (%HBM stays low, `L2served=False`, ncu confirms HBM-served + low DRAM%):
  per-CTA-bound on sm_103 at all reachable context → NVFP4 = capacity + accuracy, full stop (the
  stronger result). Update `roofline/archs.py` B300 (L2 + measured constants), `docs/results.md`/
  `decisions.md` Step 10 B300 section, `interview-prep.md`; commit the figure + output notebook.